# MVP-1 — A4: Ranking corregido + DANN condicionado

**Tres correcciones simultáneas sobre A3:**

1. **Ranking loss corregido:** usa pdl_head output (escalar), no norma L2 del embedding
2. **Todas las magnificaciones** (~760 imgs) + DANN adversarial contra magnificación (batch puro)
3. **DANN condicionado contra study_part:** adversario recibe (z, PDL, cell_line) → study_part.
   Gradient reversal solo sobre z. Objetivo: minimizar ΔAUC, no AUC raw.

**Criterios de aceptación:**
- ΔAUC incremental (study_part) ≤ 0.10
- PDL bins AUC ≥ 0.85
- Worst fold Spearman ≥ 0.60
- Fusion-readiness: correlación parcial mtDNA/telómero controlando PDL + cell_line

---

## SECCIÓN 0 — CONFIG

In [1]:
import os

# ============================================================
# >>> AJUSTAR ESTAS 4 RUTAS <<<
# ============================================================
MANIFEST_PATH = "/Users/JCB/Documentos/Proyecto Integrador/data/manifests/manifest_mvp1_finetune_20260319_153838.csv"
CSV_CENTRAL   = "/Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv"
IMAGE_ROOT    = "/Volumes/SanDisk SSD 1TB/Storage/Data/Cellular_Lifespan_Study_Brightfield_Images"
OUTPUT_DIR    = "/Users/JCB/Documentos/Proyecto Integrador/results/mvp1_A4"

# ============================================================
# COLUMNAS DEL MANIFEST
# ============================================================
COL_IMG_PATH      = "img_path"
COL_PDL_NORM      = "pdl_norm"
COL_PDL_RAW       = "pdl"
COL_FOLD          = "fold"
COL_CELL_LINE     = "cell_line"
COL_STUDY_PART    = "study_part"
COL_MAGNIFICATION = "magnification"
COL_SAMPLE_ID     = "sample_id"
COL_TELOMERE      = "telomere_length"
COL_MTDNA         = "mtdna_cn"

# ============================================================
# HIPERPARÁMETROS
# ============================================================
BATCH_SIZE    = 16
EPOCHS        = 40
PATIENCE      = 10
LR_HEAD       = 1e-3
WEIGHT_DECAY  = 1e-4
EMBEDDING_DIM = 256
IMG_SIZE      = 224
SEED          = 42

# ============================================================
# PESOS DE PÉRDIDAS
# ============================================================
LAMBDA_PDL       = 1.0
LAMBDA_RANK      = 0.3
LAMBDA_CONS      = 0.2
LAMBDA_DANN_MAG  = 0.3    # adversarial vs magnificación (batch puro — más agresivo)
LAMBDA_DANN_SP   = 0.15   # adversarial condicionado vs study_part (más conservador)
DANN_MAX_MAG     = 0.5    # máximo para magnificación
DANN_MAX_SP      = 0.3    # máximo para study_part (moderado — es mezcla bio+batch)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Config cargada. Output → {OUTPUT_DIR}")

✅ Config cargada. Output → /Users/JCB/Documentos/Proyecto Integrador/results/mvp1_A4


## SECCIÓN 1 — IMPORTS

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import json
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Function
import torchvision.transforms as T
import torchvision.models as models

from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

from PIL import Image

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("🖥  Device: Apple MPS")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🖥  Device: CUDA — {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("🖥  Device: CPU")

🖥  Device: Apple MPS


## SECCIÓN 2 — CARGAR MANIFEST (TODAS las magnificaciones)

In [3]:
df = pd.read_csv(MANIFEST_PATH)
print(f"Manifest cargado: {df.shape[0]} filas")

# Resolver rutas
sample_path = df[COL_IMG_PATH].iloc[0]
if not os.path.isabs(sample_path):
    df["_img_abs"] = df[COL_IMG_PATH].apply(lambda p: os.path.join(IMAGE_ROOT, p))
else:
    df["_img_abs"] = df[COL_IMG_PATH]

# Verificar existencia
exists_mask = df["_img_abs"].apply(os.path.exists)
df = df[exists_mask].reset_index(drop=True)
print(f"✅ {len(df)} imágenes verificadas en disco")

# SIN filtro de magnificación — usamos todas
# Solo excluir las 2 imágenes de 4x (demasiado pocas para ser una clase)
n_antes = len(df)
df = df[df[COL_MAGNIFICATION].astype(str).isin(["10", "20"])].reset_index(drop=True)
print(f"🔬 Filtro 4x: {n_antes} → {len(df)} imágenes (excluidas {n_antes - len(df)} de 4x)")

# Codificar study_part
study_parts = sorted(df[COL_STUDY_PART].unique())
sp_to_idx = {sp: i for i, sp in enumerate(study_parts)}
df["_sp_idx"] = df[COL_STUDY_PART].map(sp_to_idx)
N_STUDY_PARTS = len(study_parts)

# Codificar magnification
mags = sorted(df[COL_MAGNIFICATION].astype(str).unique())
mag_to_idx = {m: i for i, m in enumerate(mags)}
df["_mag_idx"] = df[COL_MAGNIFICATION].astype(str).map(mag_to_idx)
N_MAGS = len(mags)

# Codificar cell_line
cell_lines = sorted(df[COL_CELL_LINE].unique())
cl_to_idx = {cl: i for i, cl in enumerate(cell_lines)}
df["_cl_idx"] = df[COL_CELL_LINE].map(cl_to_idx)
N_CELL_LINES = len(cell_lines)

print(f"\n📊 Resumen:")
print(f"   Imágenes: {len(df)}")
print(f"   Cell lines: {cell_lines} ({N_CELL_LINES})")
print(f"   Folds: {df[COL_FOLD].value_counts().sort_index().to_dict()}")
print(f"   Study parts: {study_parts} ({N_STUDY_PARTS} clases)")
print(f"   Magnificaciones: {mags} ({N_MAGS} clases)")

Manifest cargado: 763 filas
✅ 763 imágenes verificadas en disco
🔬 Filtro 4x: 763 → 761 imágenes (excluidas 2 de 4x)

📊 Resumen:
   Imágenes: 761
   Cell lines: ['hFB1', 'hFB11', 'hFB12', 'hFB13', 'hFB14', 'hFB2'] (6)
   Folds: {0.0: 248, 1.0: 427, 2.0: 86}
   Study parts: [1, 2, 3, 5] (4 clases)
   Magnificaciones: ['10', '20'] (2 clases)


/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_66935/3374847510.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_img_abs"] = df[COL_IMG_PATH]
/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_66935/3374847510.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_sp_idx"] = df[COL_STUDY_PART].map(sp_to_idx)
/var/folders/fy/k8jygkfn6hd63_wrlzhk82c40000gp/T/ipykernel_66935/3374847510.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many

## SECCIÓN 3 — DATASET

In [4]:
class A4Dataset(Dataset):
    """Retorna img, img_aug, pdl, cl_idx, sp_idx, mag_idx, pdl_for_cond, cl_onehot."""

    def __init__(self, dataframe, transform, transform_aug=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.transform_aug = transform_aug or transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["_img_abs"])
        if img.mode != "RGB":
            img = img.convert("RGB")

        img1 = self.transform(img)
        img2 = self.transform_aug(img)

        pdl = torch.tensor(row[COL_PDL_NORM], dtype=torch.float32)
        cl_idx = torch.tensor(row["_cl_idx"], dtype=torch.long)
        sp_idx = torch.tensor(row["_sp_idx"], dtype=torch.long)
        mag_idx = torch.tensor(row["_mag_idx"], dtype=torch.long)

        # One-hot de cell_line para el DANN condicionado
        cl_onehot = torch.zeros(N_CELL_LINES, dtype=torch.float32)
        cl_onehot[row["_cl_idx"]] = 1.0

        return img1, img2, pdl, cl_idx, sp_idx, mag_idx, cl_onehot


train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_transform_aug = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("✅ Dataset definido")

✅ Dataset definido


## SECCIÓN 4 — MODELO A4
Gradient Reversal + DANN condicionado (study_part) + DANN directo (magnificación)

In [5]:
class GradientReversalFn(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.alpha * grad_output, None


class PDLEncoderA4(nn.Module):
    """
    Arquitectura A4:
    - ResNet-34 (frozen) → embedding 256-dim
    - PDL head (escalar — también usado para ranking loss)
    - DANN head magnificación: input = z → predice mag (batch puro)
    - DANN head study_part CONDICIONADO: input = (z, pdl, cl_onehot) → predice sp
      gradient reversal solo sobre z
    """

    def __init__(self, embedding_dim=256, n_study_parts=4, n_mags=2,
                 n_cell_lines=6, freeze_backbone=True):
        super().__init__()

        backbone = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        backbone_out = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # Embedding head
        self.embed_head = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        # PDL head (escalar — Sigmoid para [0,1])
        self.pdl_head = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

        # DANN: magnificación (batch puro — adversarial directo)
        self.dann_mag = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_mags),
        )

        # DANN: study_part CONDICIONADO
        # Input: z (reversed) + pdl (1) + cell_line_onehot (n_cell_lines)
        cond_input_dim = embedding_dim + 1 + n_cell_lines
        self.dann_sp_cond = nn.Sequential(
            nn.Linear(cond_input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_study_parts),
        )

        self.embedding_dim = embedding_dim
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"   Params entrenables: {trainable:,} / {total:,}")

    def forward(self, x, pdl_for_cond=None, cl_onehot=None,
                alpha_mag=0.0, alpha_sp=0.0):
        features = self.backbone(x)
        embedding = self.embed_head(features)
        pdl_hat = self.pdl_head(embedding).squeeze(-1)

        # DANN magnificación: adversarial directo sobre z
        z_rev_mag = GradientReversalFn.apply(embedding, alpha_mag)
        mag_logits = self.dann_mag(z_rev_mag)

        # DANN study_part condicionado: (z_reversed, pdl, cl_onehot)
        # pdl y cl_onehot NO pasan por gradient reversal — son contexto
        sp_logits = None
        if pdl_for_cond is not None and cl_onehot is not None:
            z_rev_sp = GradientReversalFn.apply(embedding, alpha_sp)
            # Detach pdl y cl_onehot para que el adversario no propague a ellos
            cond_input = torch.cat([
                z_rev_sp,
                pdl_for_cond.unsqueeze(1).detach(),
                cl_onehot.detach(),
            ], dim=1)
            sp_logits = self.dann_sp_cond(cond_input)

        return pdl_hat, embedding, mag_logits, sp_logits


print("✅ PDLEncoderA4 definido")
_ = PDLEncoderA4(EMBEDDING_DIM, N_STUDY_PARTS, N_MAGS, N_CELL_LINES)

✅ PDLEncoderA4 definido
   Params entrenables: 206,727 / 21,491,399


## SECCIÓN 5 — PÉRDIDAS CORREGIDAS

In [6]:
def ranking_loss_scalar(pdl_hat, pdl_true, cl_idx):
    """
    Ranking loss intra cell_line usando el OUTPUT ESCALAR (pdl_hat),
    NO la norma del embedding. Penaliza si pdl_hat no respeta el orden
    de pdl_true dentro del mismo donante.
    """
    loss = torch.tensor(0.0, device=pdl_hat.device)
    n_pairs = 0

    for cl in cl_idx.unique():
        mask = cl_idx == cl
        if mask.sum() < 2:
            continue

        scores = pdl_hat[mask]
        targets = pdl_true[mask]

        n = scores.shape[0]
        for i in range(n):
            for j in range(i + 1, n):
                diff = abs(targets[i] - targets[j])
                if diff < 0.05:
                    continue
                if targets[i] < targets[j]:
                    margin = diff * 0.3
                    loss += torch.relu(margin - (scores[j] - scores[i]))
                else:
                    margin = diff * 0.3
                    loss += torch.relu(margin - (scores[i] - scores[j]))
                n_pairs += 1

    return loss / max(n_pairs, 1)


def consistency_loss(emb1, emb2):
    return torch.mean((emb1 - emb2) ** 2)


print("✅ Pérdidas corregidas definidas (ranking por head escalar)")

✅ Pérdidas corregidas definidas (ranking por head escalar)


## SECCIÓN 6 — ENTRENAMIENTO

In [7]:
def dann_alpha_schedule(epoch, max_epochs, max_alpha):
    p = epoch / max_epochs
    return max_alpha * (2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)


@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    all_pdl, all_hat, all_emb = [], [], []
    for batch in loader:
        img1, img2, pdl, cl_idx, sp_idx, mag_idx, cl_onehot = batch
        img1 = img1.to(device)
        pdl_hat, emb, _, _ = model(img1, alpha_mag=0.0, alpha_sp=0.0)
        all_pdl.extend(pdl.numpy())
        all_hat.extend(pdl_hat.cpu().numpy())
        all_emb.append(emb.cpu().numpy())

    all_pdl = np.array(all_pdl)
    all_hat = np.array(all_hat)
    all_emb = np.vstack(all_emb)

    mae = np.mean(np.abs(all_pdl - all_hat))
    rho, p_val = spearmanr(all_pdl, all_hat)
    ss_res = np.sum((all_pdl - all_hat) ** 2)
    ss_tot = np.sum((all_pdl - all_pdl.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0

    return {
        "mae_norm": mae, "spearman": rho, "spearman_p": p_val, "r2": r2,
        "y_true": all_pdl, "y_pred": all_hat, "embeddings": all_emb,
    }


criterion_pdl = nn.L1Loss()
criterion_ce = nn.CrossEntropyLoss()
folds = sorted(df[COL_FOLD].unique())

print(f"Folds: {folds}")
print(f"Comenzando entrenamiento A4...")

results_per_fold = {}
all_embeddings = []

for val_fold in folds:
    print(f"\n{'='*60}")
    print(f"  A4 | FOLD {val_fold}")
    print(f"{'='*60}")

    df_train = df[df[COL_FOLD] != val_fold]
    df_val = df[df[COL_FOLD] == val_fold]
    print(f"  Train: {len(df_train)} | Val: {len(df_val)} | "
          f"Val cells: {sorted(df_val[COL_CELL_LINE].unique())}")

    ds_train = A4Dataset(df_train, train_transform, train_transform_aug)
    ds_val = A4Dataset(df_val, val_transform, val_transform)
    dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=False)

    # Baseline trivial
    trivial_mae = np.mean(np.abs(
        df_val[COL_PDL_NORM].values - df_train[COL_PDL_NORM].mean()))

    # Modelo
    model = PDLEncoderA4(EMBEDDING_DIM, N_STUDY_PARTS, N_MAGS,
                         N_CELL_LINES, freeze_backbone=True).to(DEVICE)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

    best_mae = float("inf")
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = defaultdict(float)
        n_batches = 0

        alpha_mag = dann_alpha_schedule(epoch, EPOCHS, DANN_MAX_MAG)
        alpha_sp = dann_alpha_schedule(epoch, EPOCHS, DANN_MAX_SP)

        for batch in dl_train:
            img1, img2, pdl, cl_idx, sp_idx, mag_idx, cl_onehot = batch
            img1 = img1.to(DEVICE)
            img2 = img2.to(DEVICE)
            pdl = pdl.to(DEVICE)
            cl_idx = cl_idx.to(DEVICE)
            sp_idx = sp_idx.to(DEVICE)
            mag_idx = mag_idx.to(DEVICE)
            cl_onehot = cl_onehot.to(DEVICE)

            optimizer.zero_grad()

            # Forward
            pdl_hat, emb1, mag_logits, sp_logits = model(
                img1, pdl_for_cond=pdl, cl_onehot=cl_onehot,
                alpha_mag=alpha_mag, alpha_sp=alpha_sp)

            # L_pdl
            loss_pdl = criterion_pdl(pdl_hat, pdl)
            total = LAMBDA_PDL * loss_pdl
            epoch_loss["pdl"] += loss_pdl.item()

            # L_rank (escalar — corregido)
            loss_rank = ranking_loss_scalar(pdl_hat, pdl, cl_idx)
            total += LAMBDA_RANK * loss_rank
            epoch_loss["rank"] += loss_rank.item()

            # L_cons
            _, emb2, _, _ = model(img2, alpha_mag=0.0, alpha_sp=0.0)
            loss_cons = consistency_loss(emb1, emb2)
            total += LAMBDA_CONS * loss_cons
            epoch_loss["cons"] += loss_cons.item()

            # L_dann_mag (batch puro)
            loss_mag = criterion_ce(mag_logits, mag_idx)
            total += LAMBDA_DANN_MAG * loss_mag
            epoch_loss["dann_mag"] += loss_mag.item()

            # L_dann_sp (condicionado)
            if sp_logits is not None:
                loss_sp = criterion_ce(sp_logits, sp_idx)
                total += LAMBDA_DANN_SP * loss_sp
                epoch_loss["dann_sp"] += loss_sp.item()

            total.backward()
            optimizer.step()
            n_batches += 1

        # Evaluate
        val_m = evaluate_model(model, dl_val, DEVICE)

        improved = ""
        if val_m["mae_norm"] < best_mae:
            best_mae = val_m["mae_norm"]
            patience_counter = 0
            improved = "✓"
            torch.save(model.state_dict(),
                       os.path.join(OUTPUT_DIR, f"best_A4_fold{val_fold}.pt"))
        else:
            patience_counter += 1

        if (epoch + 1) % 10 == 0 or improved or epoch == 0:
            print(f"  Ep {epoch+1:2d} | MAE={val_m['mae_norm']:.4f} | "
                  f"ρ={val_m['spearman']:.3f} | "
                  f"αm={alpha_mag:.2f} αs={alpha_sp:.2f} {improved}")

        if patience_counter >= PATIENCE:
            print(f"  Early stopping ep {epoch+1}")
            break

    # Best model
    model.load_state_dict(torch.load(
        os.path.join(OUTPUT_DIR, f"best_A4_fold{val_fold}.pt"),
        weights_only=True))
    final = evaluate_model(model, dl_val, DEVICE)
    improvement = (1 - final["mae_norm"] / trivial_mae) * 100

    # Guardar embeddings
    df_emb = df_val.copy().reset_index(drop=True)
    df_emb["y_pred"] = final["y_pred"]
    emb_array = final["embeddings"]
    emb_df = pd.DataFrame(emb_array, columns=[f"emb_{i}" for i in range(emb_array.shape[1])])
    df_emb = pd.concat([df_emb, emb_df], axis=1)
    all_embeddings.append(df_emb)

    results_per_fold[val_fold] = {
        "n_val": len(df_val),
        "cell_lines_val": sorted(df_val[COL_CELL_LINE].unique()),
        "trivial_mae": trivial_mae, "model_mae": final["mae_norm"],
        "improvement_pct": improvement, "spearman": final["spearman"],
        "spearman_p": final["spearman_p"], "r2": final["r2"],
    }

    print(f"  📊 MAE={final['mae_norm']:.4f} (mejora {improvement:+.1f}%) | "
          f"ρ={final['spearman']:.3f} | R²={final['r2']:.3f}")

df_embeddings = pd.concat(all_embeddings, ignore_index=True)
print(f"\n✅ Entrenamiento A4 completo. Embeddings: {df_embeddings.shape}")

Folds: [0.0, 1.0, 2.0]
Comenzando entrenamiento A4...

  A4 | FOLD 0.0
  Train: 513 | Val: 248 | Val cells: ['hFB12']
   Params entrenables: 206,727 / 21,491,399
  Ep  1 | MAE=0.1762 | ρ=0.653 | αm=0.00 αs=0.00 ✓
  Ep  5 | MAE=0.1489 | ρ=0.752 | αm=0.23 αs=0.14 ✓
  Ep 10 | MAE=0.2468 | ρ=0.744 | αm=0.40 αs=0.24 
  Ep 15 | MAE=0.1481 | ρ=0.739 | αm=0.47 αs=0.28 ✓
  Ep 19 | MAE=0.1454 | ρ=0.752 | αm=0.49 αs=0.29 ✓
  Ep 20 | MAE=0.2278 | ρ=0.737 | αm=0.49 αs=0.29 
  Early stopping ep 29
  📊 MAE=0.1454 (mejora +37.9%) | ρ=0.752 | R²=0.479

  A4 | FOLD 1.0
  Train: 334 | Val: 427 | Val cells: ['hFB13', 'hFB14']
   Params entrenables: 206,727 / 21,491,399
  Ep  1 | MAE=0.1737 | ρ=0.660 | αm=0.00 αs=0.00 ✓
  Ep  4 | MAE=0.1558 | ρ=0.716 | αm=0.18 αs=0.11 ✓
  Ep  7 | MAE=0.1552 | ρ=0.735 | αm=0.32 αs=0.19 ✓
  Ep 10 | MAE=0.1327 | ρ=0.748 | αm=0.40 αs=0.24 ✓
  Ep 16 | MAE=0.1279 | ρ=0.762 | αm=0.48 αs=0.29 ✓
  Ep 20 | MAE=0.1607 | ρ=0.739 | αm=0.49 αs=0.29 
  Early stopping ep 26
  📊 MAE=0.1279

## SECCIÓN 7 — RESULTADOS

In [8]:
maes = [r["model_mae"] for r in results_per_fold.values()]
spears = [r["spearman"] for r in results_per_fold.values()]
imps = [r["improvement_pct"] for r in results_per_fold.values()]

print("=" * 70)
print("  A4 — RESULTADOS 3-FOLD CV")
print("=" * 70)

for fold, r in results_per_fold.items():
    print(f"  Fold {fold}: MAE={r['model_mae']:.4f} | "
          f"ρ={r['spearman']:.3f} | R²={r['r2']:.3f} | "
          f"mejora={r['improvement_pct']:+.1f}% | "
          f"cells={', '.join(r['cell_lines_val'])}")

print(f"\n  Promedio: MAE={np.mean(maes):.4f}±{np.std(maes):.4f} | "
      f"ρ={np.mean(spears):.3f}±{np.std(spears):.3f} | "
      f"mejora={np.mean(imps):+.1f}%")
print(f"  Worst fold ρ: {min(spears):.3f}")

s_ok = "✓" if np.mean(spears) >= 0.6 else "✗"
m_ok = "✓" if np.mean(imps) >= 25 else "✗"
w_ok = "✓" if min(spears) >= 0.6 else "✗"
print(f"\n  📋 PIDA: ρ≥0.6 {s_ok} | mejora≥25% {m_ok} | worst≥0.6 {w_ok}")

  A4 — RESULTADOS 3-FOLD CV
  Fold 0.0: MAE=0.1454 | ρ=0.752 | R²=0.479 | mejora=+37.9% | cells=hFB12
  Fold 1.0: MAE=0.1279 | ρ=0.762 | R²=0.601 | mejora=+51.0% | cells=hFB13, hFB14
  Fold 2.0: MAE=0.2111 | ρ=0.399 | R²=0.081 | mejora=+10.9% | cells=hFB1, hFB11, hFB2

  Promedio: MAE=0.1615±0.0358 | ρ=0.638±0.169 | mejora=+33.3%
  Worst fold ρ: 0.399

  📋 PIDA: ρ≥0.6 ✓ | mejora≥25% ✓ | worst≥0.6 ✗


## SECCIÓN 8 — BATCH PROBE INCREMENTAL

In [9]:
print("\n" + "=" * 70)
print("  BATCH PROBE INCREMENTAL")
print("=" * 70)

emb_cols = [c for c in df_embeddings.columns if c.startswith("emb_")]
X_emb = df_embeddings[emb_cols].values

probe_results = {}

# ===== STUDY_PART (condicionado — el importante) =====
y_sp = df_embeddings[COL_STUDY_PART].values
le_sp = LabelEncoder()
y_sp_enc = le_sp.fit_transform(y_sp)
n_sp = len(np.unique(y_sp_enc))

if n_sp >= 2:
    # Modelo A: PDL + cell_line → study_part
    X_struct = pd.get_dummies(
        df_embeddings[[COL_PDL_NORM, COL_CELL_LINE]],
        columns=[COL_CELL_LINE]).values.astype(float)
    clf_a = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
    clf_a.fit(X_struct, y_sp_enc)
    auc_a = roc_auc_score(y_sp_enc, clf_a.predict_proba(X_struct),
                          multi_class="ovr", average="macro")

    # Modelo B: PDL + cell_line + z → study_part
    X_full = np.hstack([X_struct, X_emb])
    clf_b = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
    clf_b.fit(X_full, y_sp_enc)
    auc_b = roc_auc_score(y_sp_enc, clf_b.predict_proba(X_full),
                          multi_class="ovr", average="macro")

    # Modelo C: solo z → study_part
    clf_c = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
    clf_c.fit(X_emb, y_sp_enc)
    auc_c = roc_auc_score(y_sp_enc, clf_c.predict_proba(X_emb),
                          multi_class="ovr", average="macro")

    delta_sp = auc_b - auc_a
    s1 = "✅" if delta_sp <= 0.10 else "⚠️"
    s2 = "✅" if auc_c < 0.75 else "⚠️"
    print(f"\n  STUDY_PART:")
    print(f"    AUC raw z:           {auc_c:.3f} {s2}")
    print(f"    AUC struct:          {auc_a:.3f}")
    print(f"    AUC struct+z:        {auc_b:.3f}")
    print(f"    ΔAUC incremental:    {delta_sp:+.3f} {s1}")
    probe_results["study_part"] = {"raw": auc_c, "struct": auc_a,
                                   "full": auc_b, "delta": delta_sp}

# ===== MAGNIFICACIÓN (batch puro) =====
y_mag = df_embeddings[COL_MAGNIFICATION].astype(str).values
n_mag = len(np.unique(y_mag))
if n_mag >= 2:
    le_mag = LabelEncoder()
    y_mag_enc = le_mag.fit_transform(y_mag)
    clf_mag = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
    clf_mag.fit(X_emb, y_mag_enc)
    prob_mag = clf_mag.predict_proba(X_emb)
    auc_mag = roc_auc_score(y_mag_enc, prob_mag[:, 1])
    s3 = "✅" if auc_mag < 0.75 else "⚠️"
    print(f"\n  MAGNIFICACIÓN:")
    print(f"    AUC raw z:           {auc_mag:.3f} {s3}")
    probe_results["magnification"] = {"raw": auc_mag}

# ===== PDL BINS (señal biológica — debe mantenerse) =====
pdl_bins = pd.cut(df_embeddings[COL_PDL_NORM], bins=3, labels=[0, 1, 2])
clf_pdl = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
clf_pdl.fit(X_emb, pdl_bins)
auc_pdl = roc_auc_score(pdl_bins, clf_pdl.predict_proba(X_emb),
                        multi_class="ovr", average="macro")
s4 = "✅" if auc_pdl >= 0.85 else "⚠️"
print(f"\n  PDL BINS:")
print(f"    AUC:                 {auc_pdl:.3f} {s4}")
probe_results["pdl_bins"] = {"auc": auc_pdl}

# ===== CELL LINE =====
y_cl = df_embeddings[COL_CELL_LINE].values
le_cl = LabelEncoder()
y_cl_enc = le_cl.fit_transform(y_cl)
clf_cl = LogisticRegression(max_iter=2000, random_state=SEED, C=1.0)
clf_cl.fit(X_emb, y_cl_enc)
auc_cl = roc_auc_score(y_cl_enc, clf_cl.predict_proba(X_emb),
                       multi_class="ovr", average="macro")
print(f"\n  CELL LINE:")
print(f"    AUC raw z:           {auc_cl:.3f}")
probe_results["cell_line"] = {"raw": auc_cl}

with open(os.path.join(OUTPUT_DIR, "batch_probe.json"), "w") as f:
    json.dump(probe_results, f, indent=2)
print(f"\n💾 Guardado: batch_probe.json")


  BATCH PROBE INCREMENTAL

  STUDY_PART:
    AUC raw z:           0.874 ⚠️
    AUC struct:          0.846
    AUC struct+z:        0.921
    ΔAUC incremental:    +0.076 ✅

  MAGNIFICACIÓN:
    AUC raw z:           0.827 ⚠️

  PDL BINS:
    AUC:                 0.864 ✅

  CELL LINE:
    AUC raw z:           0.946

💾 Guardado: batch_probe.json


## SECCIÓN 9 — FUSION-READINESS (correlación parcial)

In [10]:
print("\n" + "=" * 70)
print("  FUSION-READINESS — Correlación parcial")
print("=" * 70)

def partial_correlation(x, y, covariates):
    reg_x = LinearRegression().fit(covariates, x)
    res_x = x - reg_x.predict(covariates)
    reg_y = LinearRegression().fit(covariates, y)
    res_y = y - reg_y.predict(covariates)
    return spearmanr(res_x, res_y)

# PCA del embedding
pca = PCA(n_components=1, random_state=SEED)
df_embeddings["emb_pc1"] = pca.fit_transform(X_emb)[:, 0]

# Agregar por sample_id
bio_cols = [COL_TELOMERE, COL_MTDNA]
agg_cols = {"emb_pc1": "mean", COL_PDL_NORM: "first", COL_CELL_LINE: "first"}
for col in bio_cols:
    if col in df_embeddings.columns:
        agg_cols[col] = "first"

sub = df_embeddings.dropna(subset=[c for c in bio_cols if c in df_embeddings.columns])

if len(sub) > 0:
    agg = sub.groupby(COL_SAMPLE_ID).agg(agg_cols).reset_index()
    cov = pd.get_dummies(agg[[COL_PDL_NORM, COL_CELL_LINE]],
                         columns=[COL_CELL_LINE]).values.astype(float)

    fusion_results = {}
    for col in bio_cols:
        if col not in agg.columns:
            continue
        mask = agg[col].notna()
        n_valid = mask.sum()
        if n_valid < 15:
            print(f"  {col}: solo {n_valid} — skip")
            continue

        x = agg.loc[mask, "emb_pc1"].values
        y = agg.loc[mask, col].values
        c = cov[mask.values]

        rho_s, p_s = spearmanr(x, y)
        rho_p, p_p = partial_correlation(x, y, c)

        sig = "***" if p_p < 0.001 else "**" if p_p < 0.01 else "*" if p_p < 0.05 else "ns"
        print(f"  {col:25s} | simple ρ={rho_s:+.3f} | parcial ρ={rho_p:+.3f} ({sig}, p={p_p:.3e}) | n={n_valid}")

        fusion_results[col] = {
            "simple_rho": float(rho_s), "simple_p": float(p_s),
            "partial_rho": float(rho_p), "partial_p": float(p_p),
            "n": int(n_valid),
        }

    # PDL sanity check
    rho_pdl, p_pdl = spearmanr(agg["emb_pc1"], agg[COL_PDL_NORM])
    print(f"  {'PDL (sanity)':25s} | ρ={rho_pdl:+.3f} (p={p_pdl:.2e})")

    with open(os.path.join(OUTPUT_DIR, "fusion_readiness.json"), "w") as f:
        json.dump(fusion_results, f, indent=2)
    print(f"\n💾 Guardado: fusion_readiness.json")
else:
    print("  ⚠️ No hay muestras con biomarcadores")


  FUSION-READINESS — Correlación parcial
  telomere_length           | simple ρ=+0.195 | parcial ρ=-0.109 (ns, p=3.449e-01) | n=77
  mtdna_cn                  | simple ρ=-0.049 | parcial ρ=+0.185 (ns, p=1.072e-01) | n=77
  PDL (sanity)              | ρ=+0.043 (p=7.09e-01)

💾 Guardado: fusion_readiness.json


## SECCIÓN 10 — COMPARACIÓN A4 vs ANTERIORES
Pega aquí los números de A1-A3 para comparar.

In [11]:
print("\n" + "=" * 70)
print("  COMPARACIÓN CON ABLATIONS ANTERIORES")
print("=" * 70)

# Resultados anteriores (hardcoded de los runs previos)
prev = {
    "A0_baseline_10x": {"spearman": 0.738, "worst": 0.616, "delta_sp": None,
                        "mag_raw": None, "mtdna_partial": "ns"},
    "A1_rank_10x":     {"spearman": 0.754, "worst": 0.617, "delta_sp": 0.148,
                        "mag_raw": None, "mtdna_partial": "0.154 ns"},
    "A2_rank_cons_10x": {"spearman": 0.736, "worst": 0.617, "delta_sp": 0.132,
                         "mag_raw": None, "mtdna_partial": "0.252 *"},
    "A3_dann_10x":     {"spearman": 0.734, "worst": 0.595, "delta_sp": 0.125,
                        "mag_raw": None, "mtdna_partial": "0.193 ns"},
}

print(f"  {'Model':22s} | {'ρ mean':>7s} | {'worst ρ':>7s} | {'Δsp':>6s} | {'mag':>5s} | mtDNA parcial")
print(f"  {'-'*22}-+-{'-'*7}-+-{'-'*7}-+-{'-'*6}-+-{'-'*5}-+-{'-'*15}")
for name, p in prev.items():
    d = f"{p['delta_sp']:+.3f}" if p['delta_sp'] is not None else "  n/a"
    m = f"{p['mag_raw']:.3f}" if p['mag_raw'] is not None else " n/a"
    print(f"  {name:22s} | {p['spearman']:7.3f} | {p['worst']:7.3f} | {d} | {m} | {p['mtdna_partial']}")

# A4 (actual)
d_sp = probe_results.get("study_part", {}).get("delta", None)
m_raw = probe_results.get("magnification", {}).get("raw", None)
mtdna_str = "n/a"
if "fusion_results" in dir() and COL_MTDNA in fusion_results:
    fr = fusion_results[COL_MTDNA]
    sig = "*" if fr["partial_p"] < 0.05 else "ns"
    mtdna_str = f"{fr['partial_rho']:+.3f} {sig}"

d_str = f"{d_sp:+.3f}" if d_sp is not None else "  n/a"
m_str = f"{m_raw:.3f}" if m_raw is not None else " n/a"
print(f"  {'>>> A4 (this run)':22s} | {np.mean(spears):7.3f} | {min(spears):7.3f} | "
      f"{d_str} | {m_str} | {mtdna_str}")

# Guardar reporte
report = {
    "timestamp": datetime.now().isoformat(),
    "model": "A4: ranking_scalar + cons + DANN_mag + DANN_sp_cond",
    "data": f"all mags (10x+20x), {len(df)} imgs",
    "results_per_fold": {str(k): v for k, v in results_per_fold.items()},
    "aggregated": {
        "mean_spearman": float(np.mean(spears)),
        "worst_spearman": float(min(spears)),
        "mean_mae": float(np.mean(maes)),
        "mean_improvement": float(np.mean(imps)),
    },
    "batch_probe": probe_results,
    "fusion_readiness": fusion_results if "fusion_results" in dir() else {},
}
with open(os.path.join(OUTPUT_DIR, "A4_report.json"), "w") as f:
    json.dump(report, f, indent=2, default=str)
print(f"\n💾 Reporte: A4_report.json")


  COMPARACIÓN CON ABLATIONS ANTERIORES
  Model                  |  ρ mean | worst ρ |    Δsp |   mag | mtDNA parcial
  -----------------------+---------+---------+--------+-------+----------------
  A0_baseline_10x        |   0.738 |   0.616 |   n/a |  n/a | ns
  A1_rank_10x            |   0.754 |   0.617 | +0.148 |  n/a | 0.154 ns
  A2_rank_cons_10x       |   0.736 |   0.617 | +0.132 |  n/a | 0.252 *
  A3_dann_10x            |   0.734 |   0.595 | +0.125 |  n/a | 0.193 ns
  >>> A4 (this run)      |   0.638 |   0.399 | +0.076 | 0.827 | +0.185 ns

💾 Reporte: A4_report.json


## SECCIÓN 11 — INTERPRETACIÓN

### Criterios de aceptación para cerrar MVP-1:

| Criterio | Umbral | ¿Pasa? |
|----------|--------|--------|
| Spearman promedio | ≥ 0.6 | ver arriba |
| Worst fold Spearman | ≥ 0.6 | ver arriba |
| ΔAUC study_part | ≤ 0.10 | ver arriba |
| AUC magnificación (raw z) | < 0.75 | ver arriba |
| PDL bins AUC | ≥ 0.85 | ver arriba |
| mtDNA correlación parcial | significativa | ver arriba |

### Si pasa → cerrar MVP-1, documentar, avanzar a MVP-2.
### Si no pasa → ajustar λ o considerar pretrain con ~2100 imgs.